# Getting the cusip returns that can be attibuted to each factor

final step before feeding into ML

In [1]:
import duckdb

new_db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\new_factor_data.duckdb"

conn = duckdb.connect(new_db_path)

conn.execute("SHOW TABLES").fetch_df()

,name
0,crsp_clean
1,cusip_factor_exposures
2,daily_factors_table
3,factor_returns_with_cusips
4,factor_top30_monthly_returns
5,monthly_base
6,monthly_factors_table
7,monthly_with_mom
8,returns_monthly


In [3]:
conn.execute("""
CREATE OR REPLACE TABLE cusip_factor_return_components AS
SELECT
  r.month_start,
  r.cusip,
             
  e.beta_market_cap * f.market_cap_month_end AS market_cap_component,
  e.beta_turnover   * f.turnover_month_avg   AS turnover_component,
  e.beta_spread     * f.spread_month_avg     AS spread_component,
  e.beta_mom_12_1   * f.mom_12_1             AS mom_12_1_component

FROM returns_monthly r
JOIN cusip_factor_exposures e
  USING (cusip)
JOIN factor_top30_monthly_returns f
  USING (month_start)
ORDER BY r.month_start, r.cusip;
""")

In [5]:
conn.execute("SELECT * FROM cusip_factor_return_components").fetch_df()

,month_start,cusip,market_cap_component,turnover_component,spread_component,mom_12_1_component
0,2002-01-01,00036020,-0.003013,-0.004782,0.006562,NaN
1,2002-01-01,00036110,-0.005055,-0.008725,0.013516,NaN
2,2002-01-01,00037520,-0.004616,-0.008368,0.012975,NaN
3,2002-01-01,00078810,-0.004340,-0.003137,-0.011669,NaN
4,2002-01-01,00086T10,-0.007013,-0.011311,0.020648,NaN
...,...,...,...,...,...,...
1956551,2023-12-01,Y8897Y23,0.068375,0.048523,0.034006,0.043587
1956552,2023-12-01,Y8900D10,0.082467,0.065126,0.047298,0.055914
1956553,2023-12-01,Y9233510,0.000132,-0.012761,-0.012609,0.006131
1956554,2023-12-01,Y9390M10,0.231920,0.149953,0.101769,0.161178
